# Migration: Dept_pkg Conversion
**Source File:** TARGET_ODI_SQL.sql  
**Target Environment:** Databricks Spark SQL / Delta Lake  
**Conversion Date:** 2024-05-22

In [ ]:
dbutils.widgets.text("V_Dept", "")
dbutils.widgets.text("ODI_SESS_NO", "")
dbutils.widgets.text("DATASOURCE_NUM_ID", "")

### ETL Parameters
SCEN_TASK_NO: 1

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_dept_param AS
SELECT 
    COALESCE(
        (SELECT date_format(last_run_dt, 'dd-MM-yy') 
         FROM workspace.odi_src.log_table1 
         WHERE pkg_name = 'Dept_pkg'), 
        date_format(current_timestamp(), 'dd-MM-yy')
    ) AS v_dept_val;


### Staging Table
SCEN_TASK_NO: 30, 40, 50, 60

In [ ]:
%sql
DROP TABLE IF EXISTS workspace.odi_src.c_0filter;


In [ ]:
%sql
CREATE TABLE workspace.odi_src.c_0filter
(
    DEPARTMENT_ID BIGINT,
    DEPARTMENT_NAME STRING,
    MANAGER_ID BIGINT,
    LOCATION_ID BIGINT,
    LAST_UPD_DT TIMESTAMP
)
USING DELTA;


In [ ]:
%sql
INSERT INTO workspace.odi_src.c_0filter
(
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
)
SELECT 
    SRC_DEPARTMENTS_1.DEPARTMENT_ID AS DEPARTMENT_ID,
    SRC_DEPARTMENTS_1.DEPARTMENT_NAME AS DEPARTMENT_NAME,
    SRC_DEPARTMENTS_1.MANAGER_ID AS MANAGER_ID,
    SRC_DEPARTMENTS_1.LOCATION_ID AS LOCATION_ID,
    SRC_DEPARTMENTS_1.LAST_UPD_DT AS LAST_UPD_DT
FROM workspace.odi_src.src_departments AS SRC_DEPARTMENTS_1
WHERE (1=1)
AND (SRC_DEPARTMENTS_1.LAST_UPD_DT >= to_date('${V_Dept}', 'dd-MM-yy'));


In [ ]:
%sql
OPTIMIZE workspace.odi_src.c_0filter;


### Flow Table
SCEN_TASK_NO: 80, 90, 100

In [ ]:
%sql
DROP TABLE IF EXISTS workspace.odi_trg.i_trg_departments_flow;


In [ ]:
%sql
CREATE TABLE workspace.odi_trg.i_trg_departments_flow
(
    DEPARTMENT_ID BIGINT,
    DEPARTMENT_NAME STRING,
    MANAGER_ID BIGINT,
    LOCATION_ID BIGINT,
    LAST_UPD_DT TIMESTAMP,
    IND_UPDATE STRING
)
USING DELTA;


In [ ]:
%sql
/* DETECTION_STRATEGY = NOT_EXISTS */
INSERT INTO workspace.odi_trg.i_trg_departments_flow
(
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT,
    IND_UPDATE
)
SELECT 
    S.DEPARTMENT_ID,
    S.DEPARTMENT_NAME,
    S.MANAGER_ID,
    S.LOCATION_ID,
    S.LAST_UPD_DT,
    S.IND_UPDATE
FROM (
    SELECT 
        FILTER_A.DEPARTMENT_ID AS DEPARTMENT_ID,
        FILTER_A.DEPARTMENT_NAME AS DEPARTMENT_NAME,
        FILTER_A.MANAGER_ID AS MANAGER_ID,
        FILTER_A.LOCATION_ID AS LOCATION_ID,
        FILTER_A.LAST_UPD_DT AS LAST_UPD_DT,
        'I' AS IND_UPDATE
    FROM workspace.odi_src.c_0filter AS FILTER_A
    WHERE (1=1)
) AS S
WHERE NOT EXISTS (
    SELECT 1 
    FROM workspace.odi_trg.trg_departments AS T
    WHERE T.DEPARTMENT_ID = S.DEPARTMENT_ID 
    AND ((T.DEPARTMENT_NAME = S.DEPARTMENT_NAME) OR (T.DEPARTMENT_NAME IS NULL AND S.DEPARTMENT_NAME IS NULL))
    AND ((T.MANAGER_ID = S.MANAGER_ID) OR (T.MANAGER_ID IS NULL AND S.MANAGER_ID IS NULL))
    AND ((T.LOCATION_ID = S.LOCATION_ID) OR (T.LOCATION_ID IS NULL AND S.LOCATION_ID IS NULL))
    AND ((T.LAST_UPD_DT = S.LAST_UPD_DT) OR (T.LAST_UPD_DT IS NULL AND S.LAST_UPD_DT IS NULL))
);


### Cleanup


In [ ]:
%sql
DROP TABLE IF EXISTS workspace.odi_src.c_0filter;


In [ ]:
%sql
DROP TABLE IF EXISTS workspace.odi_trg.i_trg_departments_flow;
